<a href="https://colab.research.google.com/github/jgte29/ucbhonorsthesis2023-2024/blob/main/SF_DA_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install firebase-admin

# Load in Data & Libraries

In [3]:
import firebase_admin
from firebase_admin import credentials
from firebase_admin import db
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from scipy import stats
import warnings
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import linear_lm
import sklearn
from sklearn.model_selection import train_test_split
from datetime import datetime
# Initialize the Firebase app with your service account key
cred = credentials.Certificate(r'PRIVATE-KEY-sf-da-analysis-firebase.json')
firebase_admin.initialize_app(cred, {
    'databaseURL': 'https://sf-da-analysis-6f156-default-rtdb.firebaseio.com/'
})

In [4]:
# For sf_prosecutions
# Get a reference to the Firebase Realtime Database
ref = db.reference('/District_Attorney_Cases_Prosecuted_2014_2023')

# Load data from the collection
sf_prosecutions = ref.get()

# Process the loaded data & converts it to a df
sf_prosecutions = pd.json_normalize(sf_prosecutions)
sf_prosecutions = sf_prosecutions.drop('id', axis=1)

# For sf_case_resolutions_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Case_Resoultions_All_Crimes_2015_2023')

# Load data from the collection
sf_case_resolutions_all = ref.get()

# Process the loaded data & converts it to a df
sf_case_resolutions_all = pd.json_normalize(sf_case_resolutions_all)
sf_case_resolutions_all = sf_case_resolutions_all.drop('id', axis=1)

# For sf_case_resolutions_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Case_Resoultions_Violent_Crimes_2015_2023')

# Load data from the collection
sf_case_resolutions_violent = ref.get()

# Process the loaded data & converts it to a df
sf_case_resolutions_violent = pd.json_normalize(sf_case_resolutions_violent)
sf_case_resolutions_violent = sf_case_resolutions_violent.drop('id', axis=1)

# For sf_da_actions_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/DA_Actions_On_Arrests_All_Crimes_2015_2023')

# Load data from the collection
sf_da_actions_all = ref.get()

# Process the loaded data & converts it to a df
sf_da_actions_all = pd.json_normalize(sf_da_actions_all)
sf_da_actions_all = sf_da_actions_all.drop('id', axis=1)

# For sf_da_actions_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/DA_Actions_On_Arrests_Violent_Crimes_2015_2023')

# Load data from the collection
sf_da_actions_violent = ref.get()

# Process the loaded data & converts it to a df
sf_da_actions_violent = pd.json_normalize(sf_da_actions_violent)
sf_da_actions_violent = sf_da_actions_violent.drop('id', axis=1)

# For sfpd_incidents_outcomes_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Outcomes_of_SFPD_Incidents_All_Crimes_2015_2023')

# Load data from the collection
sfpd_incidents_outcomes_all = ref.get()

# Process the loaded data & converts it to a df
sfpd_incidents_outcomes_all = pd.json_normalize(sfpd_incidents_outcomes_all)
sfpd_incidents_outcomes_all = sfpd_incidents_outcomes_all.drop('id', axis=1)

# For fed_off_by_crime_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_by_Type_of_Crime_All_Crimes_2015_2022')

# Load data from the collection
fed_off_by_crime_all = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_crime_all = pd.json_normalize(fed_off_by_crime_all)
fed_off_by_crime_all = fed_off_by_crime_all.drop('id', axis=1)

# For fed_off_by_crime_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_by_Type_of_Crime_Violent_Crimes_2015_2022')

# Load data from the collection
fed_off_by_crime_violent = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_crime_violent = pd.json_normalize(fed_off_by_crime_violent)
fed_off_by_crime_violent = fed_off_by_crime_violent.drop('id', axis=1)

# For fed_off_by_dist_all
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_in_Each_District_All_Crimes_2015_2022')

# Load data from the collection
fed_off_by_dist_all = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_dist_all = pd.json_normalize(fed_off_by_dist_all)
fed_off_by_dist_all = fed_off_by_dist_all.drop('id', axis=1)

# For fed_off_by_dist_violent
# Get a reference to the Firebase Realtime Database
ref = db.reference('/Federal_Offenders_in_Each_District_Violent_Crimes_2015_2022')

# Load data from the collection
fed_off_by_dist_violent = ref.get()

# Process the loaded data & converts it to a df
fed_off_by_dist_violent = pd.json_normalize(fed_off_by_dist_violent)
fed_off_by_dist_violent = fed_off_by_dist_violent.drop('id', axis=1)

In [5]:
fed_off_by_dist_all

,Circuit,District,Fiscal_Year,N,Pct
0,Grand Total,Total,2022,64157,1.000
1,DC Circuit,District of Columbia,2022,312,0.005
2,1st Circuit,Maine,2022,180,0.003
3,1st Circuit,Massachusetts,2022,466,0.007
4,1st Circuit,New Hampshire,2022,180,0.003
...,...,...,...,...,...
755,11th Circuit,"Florida, Northern",2015,255,0.004
756,11th Circuit,"Florida, Southern",2015,2238,0.032
757,11th Circuit,"Georgia, Middle",2015,377,0.005
758,11th Circuit,"Georgia, Northern",2015,592,0.008


In [6]:
date_format = "%Y-%m-%dT%H:%M:%S.%fZ"

sf_prosecutions['arrest_date'] = pd.to_datetime(sf_prosecutions['arrest_date'], format = date_format)
sf_prosecutions['data_as_of'] = pd.to_datetime(sf_prosecutions['data_as_of'], format = date_format)
sf_prosecutions['data_loaded_at'] = pd.to_datetime(sf_prosecutions['data_loaded_at'], format = date_format)

In [7]:
boudin_start_date = datetime(2020, 1, 8, 0, 0, 0)
boudin_end_date = datetime(2022, 7, 9, 0, 0, 0)
sf_prosecutions_boudin = sf_prosecutions[(sf_prosecutions["arrest_date"] >= boudin_start_date) &
                                         (sf_prosecutions["arrest_date"] < boudin_end_date)]

sf_prosecutions_boudin

,arrest_date,case_status,court_number,crime_type,da_action_taken,data_as_of,data_loaded_at,dv_case,filed_case_type,incident_number,list_of_filed_charges
9383,2020-01-09 08:00:00,Filed Motion to Revoke,2506152,"Forgery, Checks, Access Cards",New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,Z20171107-02506152,470D/F/0
9384,2020-01-30 08:00:00,Filed Motion to Revoke,2520522,Theft,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,Z20200115-02520522,"487C/F/0, 245A4/M/0"
9385,2020-01-14 08:00:00,Filed Motion to Revoke,14005420,Assault,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,Yes,MTR,140153084,"591.5/M/0, 273.5A/F/0, 273.5A/F/0, 273.5A/F/0,..."
9386,2020-01-22 08:00:00,Filed Motion to Revoke,15018930,Robbery,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,150735638,"242/M/0, 245A4/F/0, 243B/M/0, 211/F/2"
9387,2020-01-16 08:00:00,Filed Motion to Revoke,15026261,Narcotics,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,MTR,150025953,"11378/F/0, 11377/M/0, 23123A/I/0"
...,...,...,...,...,...,...,...,...,...,...,...
99090,2022-06-24 07:00:00,,2530936,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,,23-00344,
99091,2022-06-13 07:00:00,Pending,2530799,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,,Z20220613-02530799,
99092,2022-07-06 07:00:00,Fugitive Status,22006648,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,Felony,220445185,"664/1085A/F/0, 664/496DA/F/0"
99096,2022-07-04 07:00:00,Pled Guilty to Other Case or Other DA Action,22006556,,New charges filed,2023-08-28 07:00:00,2023-08-31 05:01:04,No,,220438726,


# PART I - Boudin v National Trends (Fixed Effects)

## All Crimes

In [9]:
def year_marker(year, start, end=9999, mode = 'range'):
  # Default of 9999 is used to indicate ongoing
  if mode == 'before':
    try:
      if year < start:
        return 1
      else:
        return 0
    except:
      return 0

  if mode == 'after':
    start += 1 #Increments start by 1

  # Used for both 'after' and 'range' modes
  else:
    try:
      if year >= start and year <= end:
        return 1
      else:
        return 0
    except:
      return 0

In [10]:
fed_off_totals_all = fed_off_by_dist_all[fed_off_by_dist_all["District"] == "Total"].reset_index(drop=True)
fed_off_totals_all = fed_off_totals_all.iloc[0:, 2:5].drop("Pct", axis = 1)

#Renames the column to be consistent with future dfs
fed_off_totals_all.rename(columns={'Fiscal_Year': 'Year'}, inplace=True)
fed_off_totals_all.rename(columns={'N': 'Total'}, inplace=True)

fed_off_totals_all['Log_Total'] = np.log(fed_off_totals_all['Total'])
fed_off_totals_all['Boudin'] = 0
fed_off_totals_all['SF'] = 0
fed_off_totals_all['Before_COVID'] = fed_off_totals_all.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)
fed_off_totals_all

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,64157,11.069088,0,0,0
1,2021,57304,10.956126,0,0,0
2,2020,64591,11.075830,0,0,0
3,2019,76593,11.246261,0,0,1
4,2018,69470,11.148650,0,0,1
5,2017,66899,11.110939,0,0,1
6,2016,67794,11.124229,0,0,1
7,2015,71030,11.170858,0,0,1


In [11]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sfda_convictions = sf_case_resolutions_all[["Year", "Conviction"]]
sfda_convictions["Conviction"] = sfda_convictions["Conviction"].astype(int)

#Renames the column to be consistent with future dfs
sfda_convictions.rename(columns={'Conviction': 'Total'}, inplace=True)

sfda_convictions['Log_Total'] = np.log(sfda_convictions['Total'])
sfda_convictions['Boudin'] = sfda_convictions.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sfda_convictions['SF'] = 1
sfda_convictions['Before_COVID'] = sfda_convictions.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)

sfda_convictions_proj = sfda_convictions.copy() #Creates defensive copy
sfda_convictions_proj.iloc[8, 1] = sfda_convictions_proj.iloc[9, 1]

# Drops '2023*' Row
sfda_convictions_proj = sfda_convictions_proj.drop(9)

# Drops '2023*' Row
sfda_convictions = sfda_convictions.drop(9)

sfda_convictions["Year"] = sfda_convictions["Year"].astype(int)
sfda_convictions_proj["Year"] = sfda_convictions_proj["Year"].astype(int)

# This resets warnings
warnings.resetwarnings()

In [12]:
# Concatenate vertically (along rows)
sf_fed_off_totals = pd.concat([fed_off_totals_all, sfda_convictions], axis=0)
sf_fed_off_totals_proj = pd.concat([fed_off_totals_all, sfda_convictions_proj], axis=0)

# Reset the index
sf_fed_off_totals = sf_fed_off_totals.reset_index(drop=True)
sf_fed_off_totals_proj = sf_fed_off_totals_proj.reset_index(drop=True)

sf_fed_off_totals

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,64157,11.069088,0,0,0
1,2021,57304,10.956126,0,0,0
2,2020,64591,11.075830,0,0,0
3,2019,76593,11.246261,0,0,1
4,2018,69470,11.148650,0,0,1
5,2017,66899,11.110939,0,0,1
6,2016,67794,11.124229,0,0,1
7,2015,71030,11.170858,0,0,1
8,2015,3460,8.149024,0,1,1
9,2016,3482,8.155362,0,1,1


### 2023 Inclusive

In [13]:
#2023 Raw & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.994
Model:                            GLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     526.9
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           2.29e-13
Time:                        23:19:25   Log-Likelihood:                -156.76
No. Observations:                  17   AIC:                             323.5
Df Residuals:                      12   BIC:                             327.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.418e+06   1.15e+06     -1.236   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [14]:
#2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.990
Model:                            GLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     306.0
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           5.81e-12
Time:                        23:19:59   Log-Likelihood:                 6.1667
No. Observations:                  17   AIC:                            -2.333
Df Residuals:                      12   BIC:                             1.833
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       90.2635     78.971      1.143   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [15]:
#2023 Projected & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.994
Model:                            GLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     508.3
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           2.84e-13
Time:                        23:20:23   Log-Likelihood:                -157.04
No. Observations:                  17   AIC:                             324.1
Df Residuals:                      12   BIC:                             328.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -1.47e+06   1.17e+06     -1.260   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [16]:
#2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.990
Model:                            GLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     306.0
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           5.81e-12
Time:                        23:20:42   Log-Likelihood:                 6.1667
No. Observations:                  17   AIC:                            -2.333
Df Residuals:                      12   BIC:                             1.833
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       90.2635     78.971      1.143   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


### 2023 Exclusive

In [17]:
# Drops '2023' Row
sf_fed_off_totals_no2023 = sf_fed_off_totals[sf_fed_off_totals["Year"] != 2023]
sf_fed_off_totals_no2023

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,64157,11.069088,0,0,0
1,2021,57304,10.956126,0,0,0
2,2020,64591,11.075830,0,0,0
3,2019,76593,11.246261,0,0,1
4,2018,69470,11.148650,0,0,1
5,2017,66899,11.110939,0,0,1
6,2016,67794,11.124229,0,0,1
7,2015,71030,11.170858,0,0,1
8,2015,3460,8.149024,0,1,1
9,2016,3482,8.155362,0,1,1


In [18]:
# Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.995
Model:                            GLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                     533.0
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           1.65e-12
Time:                        23:21:33   Log-Likelihood:                -146.75
No. Observations:                  16   AIC:                             303.5
Df Residuals:                      11   BIC:                             307.4
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -9.466e+05   1.16e+06     -0.817   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [19]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.996
Model:                            GLS   Adj. R-squared:                  0.995
Method:                 Least Squares   F-statistic:                     740.7
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           2.73e-13
Time:                        23:21:50   Log-Likelihood:                 13.855
No. Observations:                  16   AIC:                            -17.71
Df Residuals:                      11   BIC:                            -13.85
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       21.7086     50.642      0.429   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


## Violent Crimes

In [20]:
fed_off_totals_violent = fed_off_by_dist_violent[fed_off_by_dist_violent["District"] == "Total"].reset_index(drop=True)
fed_off_totals_violent = fed_off_totals_violent.iloc[0:, 2:5].drop("Pct", axis = 1)

#Renames the column to be consistent with future dfs
fed_off_totals_violent.rename(columns={'Fiscal_Year': 'Year'}, inplace=True)
fed_off_totals_violent.rename(columns={'N': 'Total'}, inplace=True)

fed_off_totals_violent['Log_Total'] = np.log(fed_off_totals_violent['Total'])
fed_off_totals_violent['Boudin'] = 0
fed_off_totals_violent['SF'] = 0
fed_off_totals_violent['Before_COVID'] = fed_off_totals_violent.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)
fed_off_totals_violent

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,4417,8.393216,0,0,0
1,2021,3438,8.142645,0,0,0
2,2020,3219,8.076826,0,0,0
3,2019,4326,8.372399,0,0,1
4,2018,3973,8.287277,0,0,1
5,2017,3927,8.275631,0,0,1
6,2016,3816,8.246958,0,0,1
7,2015,3835,8.251925,0,0,1


In [21]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sfda_convictions_violent = sf_case_resolutions_violent[["Year", "Conviction"]]
sfda_convictions_violent["Conviction"] = sfda_convictions_violent["Conviction"].astype(int)

#Renames the column to be consistent with future dfs
sfda_convictions_violent.rename(columns={'Conviction': 'Total'}, inplace=True)

sfda_convictions_violent['Log_Total'] = np.log(sfda_convictions_violent['Total'])
sfda_convictions_violent['Boudin'] = sfda_convictions_violent.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sfda_convictions_violent['SF'] = 1
sfda_convictions_violent['Before_COVID'] = sfda_convictions_violent.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)

sfda_convictions_violent_proj = sfda_convictions_violent.copy() #Creates defensive copy
sfda_convictions_violent_proj.iloc[8, 1] = sfda_convictions_violent_proj.iloc[9, 1]

# Drops '2023*' Row
sfda_convictions_violent_proj = sfda_convictions_violent_proj.drop(9)

# Drops '2023*' Row
sfda_convictions_violent = sfda_convictions_violent.drop(9)

sfda_convictions_violent["Year"] = sfda_convictions_violent["Year"].astype(int)
sfda_convictions_violent_proj["Year"] = sfda_convictions_violent_proj["Year"].astype(int)

# This resets warnings
warnings.resetwarnings()

In [22]:
# Concatenate vertically (along rows)
sf_fed_off_totals_violent = pd.concat([fed_off_totals_violent, sfda_convictions_violent], axis=0)
sf_fed_off_totals_violent_proj = pd.concat([fed_off_totals_violent, sfda_convictions_violent_proj], axis=0)

# Reset the index
sf_fed_off_totals_violent = sf_fed_off_totals_violent.reset_index(drop=True)
sf_fed_off_totals_violent_proj = sf_fed_off_totals_violent_proj.reset_index(drop=True)

sf_fed_off_totals_violent

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,4417,8.393216,0,0,0
1,2021,3438,8.142645,0,0,0
2,2020,3219,8.076826,0,0,0
3,2019,4326,8.372399,0,0,1
4,2018,3973,8.287277,0,0,1
5,2017,3927,8.275631,0,0,1
6,2016,3816,8.246958,0,0,1
7,2015,3835,8.251925,0,0,1
8,2015,1039,6.946014,0,1,1
9,2016,972,6.879356,0,1,1


### 2023 Inclusive

In [23]:
#2023 Raw & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.978
Model:                            GLS   Adj. R-squared:                  0.970
Method:                 Least Squares   F-statistic:                     132.0
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           8.27e-10
Time:                        23:22:50   Log-Likelihood:                -116.98
No. Observations:                  17   AIC:                             244.0
Df Residuals:                      12   BIC:                             248.1
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.469e+05   1.11e+05     -1.330   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [24]:
#2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.958
Model:                            GLS   Adj. R-squared:                  0.944
Method:                 Least Squares   F-statistic:                     68.70
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           3.62e-08
Time:                        23:23:09   Log-Likelihood:                 4.9452
No. Observations:                  17   AIC:                            0.1097
Df Residuals:                      12   BIC:                             4.276
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       46.0238     84.854      0.542   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [25]:
#2023 Projected & Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.979
Model:                            GLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     141.7
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           5.46e-10
Time:                        23:23:26   Log-Likelihood:                -116.27
No. Observations:                  17   AIC:                             242.5
Df Residuals:                      12   BIC:                             246.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.622e+05   1.06e+05     -1.531   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [27]:
#2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.958
Model:                            GLS   Adj. R-squared:                  0.944
Method:                 Least Squares   F-statistic:                     68.70
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           3.62e-08
Time:                        23:23:57   Log-Likelihood:                 4.9452
No. Observations:                  17   AIC:                            0.1097
Df Residuals:                      12   BIC:                             4.276
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       46.0238     84.854      0.542   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


### 2023 Exclusive

In [28]:
# Drops '2023' Row
sf_fed_off_totals_violent_no2023 = sf_fed_off_totals_violent[sf_fed_off_totals_violent["Year"] != 2023]
sf_fed_off_totals_violent_no2023

,Year,Total,Log_Total,Boudin,SF,Before_COVID
0,2022,4417,8.393216,0,0,0
1,2021,3438,8.142645,0,0,0
2,2020,3219,8.076826,0,0,0
3,2019,4326,8.372399,0,0,1
4,2018,3973,8.287277,0,0,1
5,2017,3927,8.275631,0,0,1
6,2016,3816,8.246958,0,0,1
7,2015,3835,8.251925,0,0,1
8,2015,1039,6.946014,0,1,1
9,2016,972,6.879356,0,1,1


In [29]:
# Total NOT Log Normalized
np.random.seed(29)
formula = "Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:                  Total   R-squared:                       0.979
Model:                            GLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     129.2
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           3.62e-09
Time:                        23:25:03   Log-Likelihood:                -109.36
No. Observations:                  16   AIC:                             228.7
Df Residuals:                      11   BIC:                             232.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.914e+05   1.12e+05     -1.709   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [30]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Year + Boudin + SF + Before_COVID" # Location FE: SF; Time FE: Before_COVID
model = smf.gls(formula=formula, data=sf_fed_off_totals_violent_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.986
Model:                            GLS   Adj. R-squared:                  0.981
Method:                 Least Squares   F-statistic:                     190.4
Date:                Sun, 24 Sep 2023   Prob (F-statistic):           4.48e-10
Time:                        23:25:19   Log-Likelihood:                 14.123
No. Observations:                  16   AIC:                            -18.25
Df Residuals:                      11   BIC:                            -14.38
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -30.7100     49.803     -0.617   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


# PART II - SF Before & After Boudin Takes Office (DID)

## All Crimes

### 2023 Inclusive

In [32]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sf_cr_all = sf_case_resolutions_all.copy()
sf_cr_all["Conviction"] = sf_cr_all["Conviction"].astype(int)

sf_cr_all['Log_Conviction'] = np.log(sf_cr_all['Conviction'])
sf_cr_all['Log_Total'] = np.log(sf_cr_all['Total'])
sf_cr_all['Boudin'] = sf_cr_all.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sf_cr_all['Before_COVID'] = sf_cr_all.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)



sf_cr_all_proj = sf_cr_all.copy() #Creates defensive copy
sf_cr_all_proj.iloc[8, 1] = sf_cr_all_proj.iloc[9, 1] #Moves 2023* to 2023 slot

# Drops '2023*' Row
sf_cr_all_proj = sf_cr_all_proj.drop(9)

# Drops '2023*' Row
sf_cr_all = sf_cr_all.drop(9)

# Addresses type issues
sf_cr_all["Year"] = sf_cr_all["Year"].astype(int)
sf_cr_all["Med_Arrest_To_Close"] = sf_cr_all["Med_Arrest_To_Close"].astype(int)
sf_cr_all["Med_Arrest_To_Close_Conviction"] = sf_cr_all["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_all["Med_Arrest_To_Close_Dismissal"] = sf_cr_all["Med_Arrest_To_Close_Dismissal"].astype(int)

sf_cr_all_proj["Year"] = sf_cr_all_proj["Year"].astype(int)
sf_cr_all_proj["Med_Arrest_To_Close"] = sf_cr_all_proj["Med_Arrest_To_Close"].astype(int)
sf_cr_all_proj["Med_Arrest_To_Close_Conviction"] = sf_cr_all_proj["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_all_proj["Med_Arrest_To_Close_Dismissal"] = sf_cr_all_proj["Med_Arrest_To_Close_Dismissal"].astype(int)

# Gets rid of 2023 Row
sf_cr_all_no2023 = sf_cr_all[sf_cr_all["Year"] != 2023]

# This resets warnings
warnings.resetwarnings()

In [39]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.926
Model:                            GLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     12.56
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0155
Time:                        23:33:18   Log-Likelihood:                 13.356
No. Observations:                   9   AIC:                            -16.71
Df Residuals:                       4   BIC:                            -15.73
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -118.8283     52.645     -2.257   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [36]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.926
Model:                            GLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     12.56
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0155
Time:                        23:31:58   Log-Likelihood:                 13.356
No. Observations:                   9   AIC:                            -16.71
Df Residuals:                       4   BIC:                            -15.73
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -118.8283     52.645     -2.257   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [40]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.948
Model:                            GLS   Adj. R-squared:                  0.897
Method:                 Least Squares   F-statistic:                     18.34
Date:                Sun, 24 Sep 2023   Prob (F-statistic):            0.00774
Time:                        23:33:31   Log-Likelihood:                 8.1767
No. Observations:                   9   AIC:                            -6.353
Df Residuals:                       4   BIC:                            -5.367
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -3.2153     93.601     -0.034   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [41]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.948
Model:                            GLS   Adj. R-squared:                  0.897
Method:                 Least Squares   F-statistic:                     18.34
Date:                Sun, 24 Sep 2023   Prob (F-statistic):            0.00774
Time:                        23:33:41   Log-Likelihood:                 8.1767
No. Observations:                   9   AIC:                            -6.353
Df Residuals:                       4   BIC:                            -5.367
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -3.2153     93.601     -0.034   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [42]:
# 2023 Raw & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.923
Model:                              GLS   Adj. R-squared:                  0.846
Method:                   Least Squares   F-statistic:                     12.01
Date:                  Sun, 24 Sep 2023   Prob (F-statistic):             0.0168
Time:                          23:34:00   Log-Likelihood:                -55.406
No. Observations:                     9   AIC:                             120.8
Df Residuals:                         4   BIC:                             121.8
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -2.932e+05    1

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [43]:
# 2023 Projected & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.923
Model:                              GLS   Adj. R-squared:                  0.846
Method:                   Least Squares   F-statistic:                     12.01
Date:                  Sun, 24 Sep 2023   Prob (F-statistic):             0.0168
Time:                          23:34:39   Log-Likelihood:                -55.406
No. Observations:                     9   AIC:                             120.8
Df Residuals:                         4   BIC:                             121.8
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -2.932e+05    1

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [50]:
# 2023 Raw & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.974
Model:                             GLS   Adj. R-squared:                  0.947
Method:                  Least Squares   F-statistic:                     36.94
Date:                 Sun, 24 Sep 2023   Prob (F-statistic):            0.00205
Time:                         23:35:42   Log-Likelihood:                -37.837
No. Observations:                    9   AIC:                             85.67
Df Residuals:                        4   BIC:                             86.66
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -3.259e+04   1.55e+04    

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [51]:
# 2023 Projected & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.974
Model:                             GLS   Adj. R-squared:                  0.947
Method:                  Least Squares   F-statistic:                     36.94
Date:                 Sun, 24 Sep 2023   Prob (F-statistic):            0.00205
Time:                         23:36:00   Log-Likelihood:                -37.837
No. Observations:                    9   AIC:                             85.67
Df Residuals:                        4   BIC:                             86.66
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -3.259e+04   1.55e+04    

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [52]:
# 2023 Raw & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.995
Model:                                        GLS   Adj. R-squared:                  0.989
Method:                             Least Squares   F-statistic:                     188.5
Date:                            Sun, 24 Sep 2023   Prob (F-statistic):           8.33e-05
Time:                                    23:36:15   Log-Likelihood:                -27.654
No. Observations:                               9   AIC:                             65.31
Df Residuals:                                   4   BIC:                             66.30
Df Model:                                       4                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [53]:
# 2023 Projected & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.995
Model:                                        GLS   Adj. R-squared:                  0.989
Method:                             Least Squares   F-statistic:                     188.5
Date:                            Sun, 24 Sep 2023   Prob (F-statistic):           8.33e-05
Time:                                    23:36:35   Log-Likelihood:                -27.654
No. Observations:                               9   AIC:                             65.31
Df Residuals:                                   4   BIC:                             66.30
Df Model:                                       4                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [54]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.975
Model:                                       GLS   Adj. R-squared:                  0.950
Method:                            Least Squares   F-statistic:                     38.98
Date:                           Sun, 24 Sep 2023   Prob (F-statistic):            0.00185
Time:                                   23:36:54   Log-Likelihood:                -33.584
No. Observations:                              9   AIC:                             77.17
Df Residuals:                                  4   BIC:                             78.15
Df Model:                                      4                                         
Covariance Type:                       nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [57]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       1.000
Model:                            GLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                     1879.
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0178
Time:                        23:38:32   Log-Likelihood:                 44.303
No. Observations:                   9   AIC:                            -72.61
Df Residuals:                       1   BIC:                            -71.03
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [58]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       1.000
Model:                            GLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                     1879.
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0178
Time:                        23:39:02   Log-Likelihood:                 44.303
No. Observations:                   9   AIC:                            -72.61
Df Residuals:                       1   BIC:                            -71.03
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [59]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.997
Model:                            GLS   Adj. R-squared:                  0.977
Method:                 Least Squares   F-statistic:                     50.31
Date:                Sun, 24 Sep 2023   Prob (F-statistic):              0.108
Time:                        23:39:20   Log-Likelihood:                 21.248
No. Observations:                   9   AIC:                            -26.50
Df Residuals:                       1   BIC:                            -24.92
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [60]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.997
Model:                            GLS   Adj. R-squared:                  0.977
Method:                 Least Squares   F-statistic:                     50.31
Date:                Sun, 24 Sep 2023   Prob (F-statistic):              0.108
Time:                        23:39:54   Log-Likelihood:                 21.248
No. Observations:                   9   AIC:                            -26.50
Df Residuals:                       1   BIC:                            -24.92
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


### 2023 Exclusive
(Note: Can't Use Before_COVID because between (2015 & 2022), Boudin & Before_COVID are **Linearly Dependent**)

In [69]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.894
Model:                            GLS   Adj. R-squared:                  0.815
Method:                 Least Squares   F-statistic:                     11.30
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0201
Time:                        23:43:40   Log-Likelihood:                 11.401
No. Observations:                   8   AIC:                            -14.80
Df Residuals:                       4   BIC:                            -14.48
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept    -117.9801     52.489     -2.248      

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [62]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.935
Model:                            GLS   Adj. R-squared:                  0.886
Method:                 Least Squares   F-statistic:                     19.22
Date:                Sun, 24 Sep 2023   Prob (F-statistic):            0.00772
Time:                        23:40:43   Log-Likelihood:                 6.7971
No. Observations:                   8   AIC:                            -5.594
Df Residuals:                       4   BIC:                            -5.276
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -2.2724     93.324     -0.024      

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [63]:
# Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.921
Model:                              GLS   Adj. R-squared:                  0.863
Method:                   Least Squares   F-statistic:                     15.64
Date:                  Sun, 24 Sep 2023   Prob (F-statistic):             0.0113
Time:                          23:40:58   Log-Likelihood:                -49.721
No. Observations:                     8   AIC:                             107.4
Df Residuals:                         4   BIC:                             107.8
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -2.928e+05   1.09e

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [66]:
# Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.969
Model:                             GLS   Adj. R-squared:                  0.945
Method:                  Least Squares   F-statistic:                     41.39
Date:                 Sun, 24 Sep 2023   Prob (F-statistic):            0.00181
Time:                         23:41:43   Log-Likelihood:                -34.104
No. Observations:                    8   AIC:                             76.21
Df Residuals:                        4   BIC:                             76.53
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -3.268e+04   1.55e+04     -2

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [70]:
# Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_no2023)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.994
Model:                                        GLS   Adj. R-squared:                  0.989
Method:                             Least Squares   F-statistic:                     211.2
Date:                            Sun, 24 Sep 2023   Prob (F-statistic):           7.37e-05
Time:                                    23:44:25   Log-Likelihood:                -25.053
No. Observations:                               8   AIC:                             58.11
Df Residuals:                                   4   BIC:                             58.42
Df Model:                                       3                                         
Covariance Type:                        nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
----------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [71]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.914
Model:                                       GLS   Adj. R-squared:                  0.862
Method:                            Least Squares   F-statistic:                     17.68
Date:                           Sun, 24 Sep 2023   Prob (F-statistic):            0.00430
Time:                                   23:44:36   Log-Likelihood:                -39.150
No. Observations:                              9   AIC:                             86.30
Df Residuals:                                  5   BIC:                             87.09
Df Model:                                      3                                         
Covariance Type:                       nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [72]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.991
Model:                            GLS   Adj. R-squared:                  0.964
Method:                 Least Squares   F-statistic:                     36.52
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0269
Time:                        23:44:56   Log-Likelihood:                 22.798
No. Observations:                   9   AIC:                            -31.60
Df Residuals:                       2   BIC:                            -30.22
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [73]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_all)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.997
Model:                            GLS   Adj. R-squared:                  0.988
Method:                 Least Squares   F-statistic:                     113.1
Date:                Sun, 24 Sep 2023   Prob (F-statistic):            0.00879
Time:                        23:45:09   Log-Likelihood:                 21.081
No. Observations:                   9   AIC:                            -28.16
Df Residuals:                       2   BIC:                            -26.78
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


## Violent Crimes

### 2023 Inclusive

In [75]:
# This temporarily disables warnings; A warning of a depreciated opperation displayed
warnings.filterwarnings('ignore')

sf_cr_violent = sf_case_resolutions_violent.copy()
sf_cr_violent["Conviction"] = sf_cr_violent["Conviction"].astype(int)

sf_cr_violent['Log_Conviction'] = np.log(sf_cr_violent['Conviction'])
sf_cr_violent['Log_Total'] = np.log(sf_cr_violent['Total'])
sf_cr_violent['Boudin'] = sf_cr_violent.apply(lambda row : year_marker(row['Year'],
                                                                             2020, 2022),
                                                    axis = 1)
sf_cr_violent['Before_COVID'] = sf_cr_violent.apply(lambda row :
                                                              year_marker(row['Year'],
                                                                          2020, mode = 'before'),
                                                              axis = 1)



sf_cr_violent_proj = sf_cr_violent.copy() #Creates defensive copy
sf_cr_violent_proj.iloc[8, 1] = sf_cr_violent_proj.iloc[9, 1] #Moves 2023* to 2023 slot

# Drops '2023*' Row
sf_cr_violent_proj = sf_cr_violent_proj.drop(9)

# Drops '2023*' Row
sf_cr_violent = sf_cr_violent.drop(9)

# Addresses type issues
sf_cr_violent["Year"] = sf_cr_violent["Year"].astype(int)
sf_cr_violent["Med_Arrest_To_Close"] = sf_cr_violent["Med_Arrest_To_Close"].astype(int)
sf_cr_violent["Med_Arrest_To_Close_Conviction"] = sf_cr_violent["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_violent["Med_Arrest_To_Close_Dismissal"] = sf_cr_violent["Med_Arrest_To_Close_Dismissal"].astype(int)

sf_cr_violent_proj["Year"] = sf_cr_violent_proj["Year"].astype(int)
sf_cr_violent_proj["Med_Arrest_To_Close"] = sf_cr_violent_proj["Med_Arrest_To_Close"].astype(int)
sf_cr_violent_proj["Med_Arrest_To_Close_Conviction"] = sf_cr_violent_proj["Med_Arrest_To_Close_Conviction"].astype(int)
sf_cr_violent_proj["Med_Arrest_To_Close_Dismissal"] = sf_cr_violent_proj["Med_Arrest_To_Close_Dismissal"].astype(int)

# Gets rid of 2023 Row
sf_cr_violent_no2023 = sf_cr_violent[sf_cr_violent["Year"] != 2023]

# This resets warnings
warnings.resetwarnings()

In [78]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.918
Model:                            GLS   Adj. R-squared:                  0.837
Method:                 Least Squares   F-statistic:                     11.23
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0190
Time:                        23:50:13   Log-Likelihood:                 13.774
No. Observations:                   9   AIC:                            -17.55
Df Residuals:                       4   BIC:                            -16.56
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -145.4370     50.255     -2.894   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [79]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.918
Model:                            GLS   Adj. R-squared:                  0.837
Method:                 Least Squares   F-statistic:                     11.23
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0190
Time:                        23:50:21   Log-Likelihood:                 13.774
No. Observations:                   9   AIC:                            -17.55
Df Residuals:                       4   BIC:                            -16.56
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     -145.4370     50.255     -2.894   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [80]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.945
Model:                            GLS   Adj. R-squared:                  0.891
Method:                 Least Squares   F-statistic:                     17.34
Date:                Sun, 24 Sep 2023   Prob (F-statistic):            0.00860
Time:                        23:50:32   Log-Likelihood:                 9.0607
No. Observations:                   9   AIC:                            -8.121
Df Residuals:                       4   BIC:                            -7.135
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -17.3922     84.846     -0.205   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [81]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.945
Model:                            GLS   Adj. R-squared:                  0.891
Method:                 Least Squares   F-statistic:                     17.34
Date:                Sun, 24 Sep 2023   Prob (F-statistic):            0.00860
Time:                        23:50:41   Log-Likelihood:                 9.0607
No. Observations:                   9   AIC:                            -8.121
Df Residuals:                       4   BIC:                            -7.135
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      -17.3922     84.846     -0.205   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [82]:
# 2023 Raw & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.957
Model:                              GLS   Adj. R-squared:                  0.914
Method:                   Least Squares   F-statistic:                     22.34
Date:                  Sun, 24 Sep 2023   Prob (F-statistic):            0.00535
Time:                          23:50:52   Log-Likelihood:                -42.586
No. Observations:                     9   AIC:                             95.17
Df Residuals:                         4   BIC:                             96.16
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.174e+05   2.

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [83]:
# 2023 Projected & Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.957
Model:                              GLS   Adj. R-squared:                  0.914
Method:                   Least Squares   F-statistic:                     22.34
Date:                  Sun, 24 Sep 2023   Prob (F-statistic):            0.00535
Time:                          23:51:06   Log-Likelihood:                -42.586
No. Observations:                     9   AIC:                             95.17
Df Residuals:                         4   BIC:                             96.16
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -1.174e+05   2.

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [84]:
# 2023 Raw & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.944
Model:                             GLS   Adj. R-squared:                  0.888
Method:                  Least Squares   F-statistic:                     16.82
Date:                 Sun, 24 Sep 2023   Prob (F-statistic):            0.00910
Time:                         23:51:15   Log-Likelihood:                -43.391
No. Observations:                    9   AIC:                             96.78
Df Residuals:                        4   BIC:                             97.77
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -4.694e+04   2.88e+04    

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [85]:
# 2023 Projected & Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.944
Model:                             GLS   Adj. R-squared:                  0.888
Method:                  Least Squares   F-statistic:                     16.82
Date:                 Sun, 24 Sep 2023   Prob (F-statistic):            0.00910
Time:                         23:51:36   Log-Likelihood:                -43.391
No. Observations:                    9   AIC:                             96.78
Df Residuals:                        4   BIC:                             97.77
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept    -4.694e+04   2.88e+04    

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [86]:
# 2023 Raw & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.965
Model:                                        GLS   Adj. R-squared:                  0.929
Method:                             Least Squares   F-statistic:                     27.18
Date:                            Sun, 24 Sep 2023   Prob (F-statistic):            0.00369
Time:                                    23:52:00   Log-Likelihood:                -40.698
No. Observations:                               9   AIC:                             91.40
Df Residuals:                                   4   BIC:                             92.38
Df Model:                                       4                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [87]:
# 2023 Projected & Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.965
Model:                                        GLS   Adj. R-squared:                  0.929
Method:                             Least Squares   F-statistic:                     27.18
Date:                            Sun, 24 Sep 2023   Prob (F-statistic):            0.00369
Time:                                    23:52:10   Log-Likelihood:                -40.698
No. Observations:                               9   AIC:                             91.40
Df Residuals:                                   4   BIC:                             92.38
Df Model:                                       4                                         
Covariance Type:                        nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
---------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [88]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year + Before_COVID + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.954
Model:                                       GLS   Adj. R-squared:                  0.908
Method:                            Least Squares   F-statistic:                     20.84
Date:                           Sun, 24 Sep 2023   Prob (F-statistic):            0.00610
Time:                                   23:52:21   Log-Likelihood:                -40.196
No. Observations:                              9   AIC:                             90.39
Df Residuals:                                  4   BIC:                             91.38
Df Model:                                      4                                         
Covariance Type:                       nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [89]:
# 2023 Raw & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.969
Model:                            GLS   Adj. R-squared:                  0.751
Method:                 Least Squares   F-statistic:                     4.447
Date:                Sun, 24 Sep 2023   Prob (F-statistic):              0.350
Time:                        23:52:43   Log-Likelihood:                 18.120
No. Observations:                   9   AIC:                            -20.24
Df Residuals:                       1   BIC:                            -18.66
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [90]:
# 2023 Projected & Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.969
Model:                            GLS   Adj. R-squared:                  0.751
Method:                 Least Squares   F-statistic:                     4.447
Date:                Sun, 24 Sep 2023   Prob (F-statistic):              0.350
Time:                        23:52:54   Log-Likelihood:                 18.120
No. Observations:                   9   AIC:                            -20.24
Df Residuals:                       1   BIC:                            -18.66
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [91]:
# 2023 Raw & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.994
Model:                            GLS   Adj. R-squared:                  0.950
Method:                 Least Squares   F-statistic:                     22.70
Date:                Sun, 24 Sep 2023   Prob (F-statistic):              0.160
Time:                        23:53:04   Log-Likelihood:                 18.805
No. Observations:                   9   AIC:                            -21.61
Df Residuals:                       1   BIC:                            -20.03
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [92]:
# 2023 Projected & Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Before_COVID + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.994
Model:                            GLS   Adj. R-squared:                  0.950
Method:                 Least Squares   F-statistic:                     22.70
Date:                Sun, 24 Sep 2023   Prob (F-statistic):              0.160
Time:                        23:53:16   Log-Likelihood:                 18.805
No. Observations:                   9   AIC:                            -21.61
Df Residuals:                       1   BIC:                            -20.03
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


### 2023 Exclusive
(Note: Can't Use Before_COVID because between (2015 & 2022), Boudin & Before_COVID are **Linearly Dependent**)

In [93]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.889
Model:                            GLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     10.73
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0220
Time:                        23:53:55   Log-Likelihood:                 11.773
No. Observations:                   8   AIC:                            -15.55
Df Residuals:                       4   BIC:                            -15.23
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept    -144.5860     50.105     -2.886      

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [94]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.921
Model:                            GLS   Adj. R-squared:                  0.862
Method:                 Least Squares   F-statistic:                     15.60
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0113
Time:                        23:54:05   Log-Likelihood:                 7.5828
No. Observations:                   8   AIC:                            -7.166
Df Residuals:                       4   BIC:                            -6.848
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept     -16.4114     84.594     -0.194      

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [95]:
# Successful_Diversion Not Log Normalized
np.random.seed(29)
formula = "Successful_Diversion ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                             
Dep. Variable:     Successful_Diversion   R-squared:                       0.953
Model:                              GLS   Adj. R-squared:                  0.918
Method:                   Least Squares   F-statistic:                     27.14
Date:                  Sun, 24 Sep 2023   Prob (F-statistic):            0.00405
Time:                          23:54:16   Log-Likelihood:                -38.326
No. Observations:                     8   AIC:                             84.65
Df Residuals:                         4   BIC:                             84.97
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -1.172e+05   2.63e

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [96]:
# Med_Arrest_To_Close Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                             GLS Regression Results                            
Dep. Variable:     Med_Arrest_To_Close   R-squared:                       0.940
Model:                             GLS   Adj. R-squared:                  0.894
Method:                  Least Squares   F-statistic:                     20.71
Date:                 Sun, 24 Sep 2023   Prob (F-statistic):            0.00672
Time:                         23:54:25   Log-Likelihood:                -39.041
No. Observations:                    8   AIC:                             86.08
Df Residuals:                        4   BIC:                             86.40
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept     -4.7e+04   2.87e+04     -1

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [97]:
# Med_Arrest_To_Close_Conviction Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Conviction ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_no2023)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                  
Dep. Variable:     Med_Arrest_To_Close_Conviction   R-squared:                       0.962
Model:                                        GLS   Adj. R-squared:                  0.934
Method:                             Least Squares   F-statistic:                     33.77
Date:                            Sun, 24 Sep 2023   Prob (F-statistic):            0.00267
Time:                                    23:54:33   Log-Likelihood:                -36.648
No. Observations:                               8   AIC:                             81.30
Df Residuals:                                   4   BIC:                             81.61
Df Model:                                       3                                         
Covariance Type:                        nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
----------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=8
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [98]:
# 2023 Projected & Med_Arrest_To_Close_Dismissal Not Log Normalized
np.random.seed(29)
formula = "Med_Arrest_To_Close_Dismissal ~ Boudin + Year + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent_proj)
result = model.fit()
print(result.summary())

                                  GLS Regression Results                                 
Dep. Variable:     Med_Arrest_To_Close_Dismissal   R-squared:                       0.872
Model:                                       GLS   Adj. R-squared:                  0.796
Method:                            Least Squares   F-statistic:                     11.40
Date:                           Sun, 24 Sep 2023   Prob (F-statistic):             0.0113
Time:                                   23:54:45   Log-Likelihood:                -44.808
No. Observations:                              9   AIC:                             97.62
Df Residuals:                                  5   BIC:                             98.40
Df Model:                                      3                                         
Covariance Type:                       nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [99]:
# Total Log Normalized
np.random.seed(29)
formula = "Log_Total ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:              Log_Total   R-squared:                       0.967
Model:                            GLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                     9.694
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0965
Time:                        23:54:58   Log-Likelihood:                 17.823
No. Observations:                   9   AIC:                            -21.65
Df Residuals:                       2   BIC:                            -20.27
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


In [100]:
# Conviction Log Normalized
np.random.seed(29)
formula = "Log_Conviction ~ Boudin + Year + Med_Arrest_To_Close + Med_Arrest_To_Close_Conviction + Med_Arrest_To_Close_Dismissal + Boudin*Year"
model = smf.gls(formula=formula, data=sf_cr_violent)
result = model.fit()
print(result.summary())

                            GLS Regression Results                            
Dep. Variable:         Log_Conviction   R-squared:                       0.993
Model:                            GLS   Adj. R-squared:                  0.974
Method:                 Least Squares   F-statistic:                     50.71
Date:                Sun, 24 Sep 2023   Prob (F-statistic):             0.0195
Time:                        23:55:07   Log-Likelihood:                 18.610
No. Observations:                   9   AIC:                            -23.22
Df Residuals:                       2   BIC:                            -21.84
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "
